<a href="https://colab.research.google.com/github/lucas-estanislau/pe-de-meia/blob/Teste-de-otimiza%C3%A7%C3%A3o-da-coleta/pe_de_meia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pé-de-Meia**
### e sua relação com a permanência e o desempenho escolar no ensino médio no Brasil

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO

In [2]:
mar24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202403"

# Visulizando os dados do mês de março de 2024 para explorar

response = requests.get(mar24, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
  arquivo = zip_ref.namelist()[0]

  with zip_ref.open(arquivo) as csv:
    df = pd.read_csv(csv, sep=";", encoding="latin1")

df

/tmp/ipykernel_4695/2082999193.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv, sep=";", encoding="latin1")


,MÊS FOLHA,MÊS REFERÊNCIA,UF,CÓDIGO MUNICÍPIO SIAFI,NOME MUNICÍPIO,NIS BENEFICIÁRIO,CPF BENEFICIÁRIO,NOME BENEFICIÁRIO,NIS RESPONSÁVEL,CPF RESPONSÁVEL,NOME RESPONSÁVEL,CÓDIGO TIPO INCENTIVO,TIPO INCENTIVO,PERÍODO INCENTIVO,CÓDIGO ETAPA ENSINO,ETAPA ENSINO,DATA DO PAGAMENTO,VALOR PARCELA
0,202403,202403,AC,643,ACRELANDIA,2.126185e+10,***.408.142-**,ALLAN JHONATA DA SILVA HERMOGENES,1.624864e+10,***.338.912-**,IRACILDA HERMOGENES DE SOUZA FERREIRA,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,26/03/2024,"200,00"
1,202403,202403,AC,643,ACRELANDIA,2.120773e+10,***.655.722-**,BIANCA DA SILVA RIBEIRO,1.609672e+10,***.872.892-**,ANTONIA ROSEMIRA SANTOS DA SILVA,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,02/04/2024,"200,00"
2,202403,202403,AC,643,ACRELANDIA,2.380363e+10,***.379.332-**,DAYANE CARVALHO DA SILVA,2.102097e+10,***.323.582-**,ANGELA MARIA CARVALHO,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,03/04/2024,"200,00"
3,202403,202403,AC,643,ACRELANDIA,2.122791e+10,***.703.662-**,DEBORA DANTAS DE LIMA,2.122791e+10,***.703.662-**,DEBORA DANTAS DE LIMA,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,03/04/2024,"200,00"
4,202403,202403,AC,643,ACRELANDIA,2.362701e+10,***.319.892-**,DENNIS BOONE DE CARVALHO,2.093918e+10,***.434.512-**,NELSI BOONE,1,Matrícula,Ano letivo 2024,45,EJA - Educação de Jovens e Adultos,28/03/2024,"200,00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2416247,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.032558e+10,***.326.831-**,ANA LIGIA OLIVEIRA SANTOS,1,Matrícula,Ano letivo 2024,27,3º ano - Ensino Médio Regular,02/04/2024,"200,00"
2416248,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.073483e+10,***.793.551-**,JUCINEIA MOREIRA DA SILVA,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,01/04/2024,"200,00"
2416249,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.121601e+10,***.916.451-**,ANA FRANCISCA BEZERRA DA CONCEICAO,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,26/03/2024,"200,00"
2416250,202403,202403,TO,9643,XAMBIOA,NaN,NaN,*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2.063251e+10,***.447.351-**,KEILIANY SOARES BORGES,1,Matrícula,Ano letivo 2024,25,1º ano - Ensino Médio Regular,27/03/2024,"200,00"


In [3]:
df.isna().sum()

,0
MÊS FOLHA,0
MÊS REFERÊNCIA,0
UF,0
CÓDIGO MUNICÍPIO SIAFI,0
NOME MUNICÍPIO,0
NIS BENEFICIÁRIO,2147264
CPF BENEFICIÁRIO,2147193
NOME BENEFICIÁRIO,0
NIS RESPONSÁVEL,158758
CPF RESPONSÁVEL,158192


Há 2.147.193 dados ausentes no campo "CPF BENEFICIÁRIO". Ao observar o dataframe, percebe-se que os valores NaN nesse campo possuem o campo "NOME BENEFICIÁRIO" contendo  "*** BENEFICIÁRIO MENOR DE 18 ANOS ***"

In [5]:
# Conferindo valores possíveis quando o campo "CPF BENENFICIÁRIO" é NaN
df.loc[
    df['CPF BENEFICIÁRIO'].isna(),
    'NOME BENEFICIÁRIO'
].value_counts()

,count
NOME BENEFICIÁRIO,
*** BENEFICIÁRIO MENOR DE 18 ANOS ***,2147193


Esse resultado mostra que todos os valores ausentes no campo de CPF refere-se a beneficiários menores de 18 anos.

In [7]:
# Verificando os tipos de incentivos existentes
df['TIPO INCENTIVO'].value_counts()

,count
TIPO INCENTIVO,
Matrícula,2416252


In [8]:
df['VALOR PARCELA'].value_counts()

,count
VALOR PARCELA,
"200,00",2416252


Com isso, é possível concluir que todos os dados de março de 2024 são com incentivo a MATRÍCULA com valor de 200,00 por pagamento.

In [11]:
# Verificando quantidade de linhas duplicadas
df.duplicated().sum()

In [16]:
duplicados = df.duplicated()

In [17]:
df.loc[duplicados, 'CPF BENEFICIÁRIO'].isna().value_counts()

,count
CPF BENEFICIÁRIO,
True,106641


Todas as 106.641 linhas identificadas como duplicadas pertencem a registros de beneficiários com CPF ausente.

Total de registros: 2.416.251

Registros com CPF ausente: 2.147.193

Linhas identificadas como duplicadas: 106.641

Linhas duplicadas com CPF ausente: 106.641

Linhas duplicadas com CPF preenchido: 0

Com isso, a quantidade de pagamentos pode ser uma aproximação da quantidade de beneficiários atendidos naquele período, mas não é possível afirmar que são exatamente iguais. Portanto, não podemos descartar estas duplicatas.

#Processamento

In [ ]:
# Carrega os dados zipados de cada mês do programa dentro do Portal da Transparência
def carregar_mes(url):
  response = requests.get(url, timeout=120)
  response.raise_for_status()

  with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
      arquivo = zip_ref.namelist()[0]

      with zip_ref.open(arquivo) as csv:
          df = pd.read_csv(csv, sep=";", encoding="latin1", usecols=["UF", "VALOR PARCELA"])

  return df

# Converte 'VALOR PARCELA' de object para float
def converter_valor(df):
  df['VALOR PARCELA'] = (
      df['VALOR PARCELA'].str.replace(",", ".", regex=False).astype(float)
  )
  return df

# Coleta os dados do referido mês, trata a coluna de 'VALOR PARCELA' e agrupar por UF. Devolve um df resumido e agrupado
def coletar_mes(url):
  df_mes = carregar_mes(url)
  converter_valor(df_mes)
  df_resumido = df_mes.groupby('UF', as_index=False)['VALOR PARCELA'].sum().reset_index().sort_values('UF') # Inserir quantidade de beneficiários por estado

  return df_resumido

# Processa o mês em um dataframe agrupado e salva o csv no Google Drive
def processar_ano(urls, dfs, ano):
    for mes, url in urls.items():
        dfs[mes] = coletar_mes(url)

        dfs[mes].to_csv(
            f"/content/drive/MyDrive/pe-de-meia/pe-de-meia-data/{ano}/{mes}_uf.csv",
            index=False
        )

        print(f"{mes} processado e salvo com sucesso!")

    return dfs

In [ ]:
pdm2024 = {
    "mar24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202403",
    "abr24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202404",
    "mai24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202405",
    "jun24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202406",
    "ago24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202408",
    "set24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202409",
    "out24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202410",
    "nov24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202411",
    "dez24": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202412"
}

In [ ]:
pdm2025 = {
    "jan25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202501",
    "fev25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202502",
    "mar25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202503",
    "abr25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202504",
    "mai25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202505",
    "jun25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202506",
    "jul25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202507",
    "ago25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202508",
    "set25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202509",
    "out25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202510",
    "nov25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202511",
    "dez25": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202512"
}

In [ ]:
pdm2026 = {
    "fev26": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202602",
    "mar26": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202603",
    "abr26": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202604",
    "mai26": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202605",
    "jun26": "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202606"
}

In [ ]:
# Dicionário com os dataframes agrupados
dfs2024 = {}
dfs2025 = {}
dfs2026 = {}

processar_ano(pdm2024, dfs2024, 2024)
processar_ano(pdm2025, dfs2025, 2025)
processar_ano(pdm2026, dfs2026, 2026)

mar24 processado e salvo com sucesso!
abr24 processado e salvo com sucesso!
mai24 processado e salvo com sucesso!
jun24 processado e salvo com sucesso!
ago24 processado e salvo com sucesso!
set24 processado e salvo com sucesso!
out24 processado e salvo com sucesso!
nov24 processado e salvo com sucesso!
dez24 processado e salvo com sucesso!
jan25 processado e salvo com sucesso!
fev25 processado e salvo com sucesso!
mar25 processado e salvo com sucesso!
abr25 processado e salvo com sucesso!
mai25 processado e salvo com sucesso!
jun25 processado e salvo com sucesso!
jul25 processado e salvo com sucesso!
ago25 processado e salvo com sucesso!
set25 processado e salvo com sucesso!
out25 processado e salvo com sucesso!
nov25 processado e salvo com sucesso!
dez25 processado e salvo com sucesso!
fev26 processado e salvo com sucesso!
mar26 processado e salvo com sucesso!
abr26 processado e salvo com sucesso!
mai26 processado e salvo com sucesso!
jun26 processado e salvo com sucesso!


{'fev26':     index  UF  VALOR PARCELA
 0       0  AC     18702025.0
 1       1  AL     71987250.0
 2       2  AM    125125625.0
 3       3  AP     20314400.0
 4       4  BA    257040250.0
 5       5  CE    173235925.0
 6       6  DF     16533350.0
 7       7  ES     53022250.0
 8       8  GO     73206350.0
 9       9  MA    176819400.0
 10     10  MG    284486225.0
 11     11  MS     25798875.0
 12     12  MT     36975750.0
 13     13  PA    216973225.0
 14     14  PB     93162400.0
 15     15  PE    193488500.0
 16     16  PI     73396800.0
 17     17  PR    109965150.0
 18     18  RJ    209043975.0
 19     19  RN     63457800.0
 20     20  RO     26421150.0
 21     21  RR      5547050.0
 22     22  RS     85604375.0
 23     23  SC     45285625.0
 24     24  SE     52584625.0
 25     25  SP    297381775.0
 26     26  TO     23480925.0,
 'mar26':     index  UF  VALOR PARCELA
 0       0  AC      7207425.0
 1       1  AL     29032900.0
 2       2  AM     31688100.0
 3       3  AP      5